# Stage 1: vLLM serving on a free Colab T4

Serves Qwen2.5-0.5B-Instruct with vLLM and loads it with `vllm bench serve` from the same machine:
512 random input tokens, 128 output tokens, concurrency 1, 4, 16, 32, three repeats in one session.
Run the cells top to bottom (Runtime > Change runtime type > T4 GPU). The last cell prints every result file;
copy that block into `bench/results/<date>_colab-t4/` and run `bench/summarize.py` on it.

In [2]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Python 3.13.16


In [3]:
!pip install -q vllm --extra-index-url https://download.pytorch.org/whl/cu129
!pip list 2>/dev/null | grep -i -E "^(vllm|torch|transformers) "
# the flags the benchmark cell passes must all be listed here
!vllm bench serve --help | grep -o -E -- "--(backend|base-url|dataset-name|ignore-eos|seed|result-dir|result-filename|metric-percentiles|max-concurrency|random-input-len|random-output-len|num-prompts|save-result)\b" | sort -u

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 whic

In [5]:
!vllm bench serve --help=all 2>&1 | grep -o -E -- "--(backend|base-url|dataset-name|ignore-eos|seed|result-dir|result-filename|metric-percentiles|max-concurrency|random-input-len|random-output-len|num-prompts|save-result|model|percentile-metrics|request-rate|num-warmups|endpoint)\b" | sort -u

--backend
--base-url
--dataset-name
--endpoint
--ignore-eos
--max-concurrency
--metric-percentiles
--model
--num-prompts
--num-warmups
--percentile-metrics
--random-input-len
--random-output-len
--request-rate
--result-dir
--result-filename
--save-result
--seed


In [6]:
import subprocess, time, urllib.request

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# --dtype half: the model ships as bfloat16 and the T4 (compute capability 7.5) has no bfloat16
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "4096"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        break
    except OSError:
        if server.poll() is not None:
            break
        time.sleep(10)
!tail -n 12 server.log
assert server.poll() is None, "vllm serve exited, see server.log"
urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
print("server ready")

(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /v1/messages/render, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /v1/completions/render, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /v1/responses/render, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /v1/chat/completions/derender, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /v1/completions/derender, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /inference/v1/generate, Methods: POST
(APIServer pid=3481) INFO 11-21 06:59:32 [launcher.py:197] Route: /inference/v1/abort_requests, Methods: POST
INFO:     Started server process [3481]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     127.0.0.1:57992 - "G

In [7]:
import itertools, json, os

os.makedirs("results", exist_ok=True)

def bench(name, concurrency, prompts, seed):
    out = f"results/{name}.json"
    if not os.path.exists(out):  # a cut session resumes at the first missing file
        with open(f"results/{name}.log", "w") as log:
            subprocess.run(
                ["vllm", "bench", "serve", "--backend", "openai", "--base-url", "http://localhost:8000",
                 "--model", MODEL, "--dataset-name", "random",
                 "--random-input-len", "512", "--random-output-len", "128", "--ignore-eos",
                 "--num-prompts", str(prompts), "--max-concurrency", str(concurrency),
                 # a new seed per run: identical prompts would hit the server's prefix cache
                 "--seed", str(seed), "--metric-percentiles", "50,95",
                 "--save-result", "--result-dir", "results", "--result-filename", f"{name}.json"],
                stdout=log, stderr=subprocess.STDOUT, check=True)
    r = json.load(open(out))
    print(name, {k: round(v, 1) for k, v in r.items()
                 if k in ("completed", "duration", "output_throughput", "median_ttft_ms", "p95_ttft_ms")})

bench("warmup", 4, 64, 1)  # discarded
for repeat, concurrency in itertools.product((1, 2, 3), (1, 4, 16, 32)):
    bench(f"c{concurrency:02d}_r{repeat}", concurrency, 256, 100 * repeat + concurrency)

warmup {'duration': 17.2, 'completed': 64, 'output_throughput': 476.8, 'median_ttft_ms': 166.6, 'p95_ttft_ms': 872.0}
c01_r1 {'duration': 228.2, 'completed': 256, 'output_throughput': 143.6, 'median_ttft_ms': 58.5, 'p95_ttft_ms': 63.5}
c04_r1 {'duration': 67.9, 'completed': 256, 'output_throughput': 482.7, 'median_ttft_ms': 177.3, 'p95_ttft_ms': 193.7}
c16_r1 {'duration': 32.4, 'completed': 256, 'output_throughput': 1010.6, 'median_ttft_ms': 411.8, 'p95_ttft_ms': 701.5}
c32_r1 {'duration': 25.7, 'completed': 256, 'output_throughput': 1273.4, 'median_ttft_ms': 495.8, 'p95_ttft_ms': 990.6}
c01_r2 {'duration': 229.3, 'completed': 256, 'output_throughput': 142.9, 'median_ttft_ms': 59.7, 'p95_ttft_ms': 64.2}
c04_r2 {'duration': 67.9, 'completed': 256, 'output_throughput': 482.9, 'median_ttft_ms': 178.9, 'p95_ttft_ms': 194.1}
c16_r2 {'duration': 32.2, 'completed': 256, 'output_throughput': 1018.4, 'median_ttft_ms': 402.0, 'p95_ttft_ms': 652.7}
c32_r2 {'duration': 25.6, 'completed': 256, 'out

In [8]:
import glob

# the server's own first-token histogram, to cross-check the client numbers above
!curl -s localhost:8000/metrics | grep -E "^vllm:time_to_first_token_seconds_(sum|count)"
print("<<<RESULTS")
for path in sorted(glob.glob("results/*.json")):
    print(json.dumps({"file": os.path.basename(path), **json.load(open(path))}))
print("RESULTS>>>")

vllm:time_to_first_token_seconds_count{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 3136.0
vllm:time_to_first_token_seconds_sum{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 834.2088704109192
<<<RESULTS
{"file": "c01_r1.json", "date": "20261121-070427", "endpoint_type": "openai", "backend": "openai", "label": null, "model_id": "Qwen/Qwen2.5-0.5B-Instruct", "tokenizer_id": "Qwen/Qwen2.5-0.5B-Instruct", "num_prompts": 256, "request_rate": "inf", "burstiness": 1.0, "max_concurrency": 1, "duration": 228.24286612499998, "completed": 256, "failed": 0, "total_input_tokens": 131072, "total_output_tokens": 32768, "request_throughput": 1.121612273567396, "request_goodput": null, "output_throughput": 143.56637101662668, "total_token_throughput": 717.8318550831334, "latencies": [0.9220138109999425, 0.8490532260000236, 0.8545713879999539, 0.8515706370000089, 0.850049522000063, 0.8600304290000622, 0.8596674309999344, 0.8625179680000201, 0.8583058829999572, 0.8584560959999408, 0.85849671

In [9]:
print("<<<COMPACT"); [print(json.dumps({k: v for k, v in {"file": os.path.basename(p), **json.load(open(p))}.items() if not isinstance(v, list)})) for p in sorted(glob.glob("results/*.json"))]; print("COMPACT>>>")

<<<COMPACT
{"file": "c01_r1.json", "date": "20261121-070427", "endpoint_type": "openai", "backend": "openai", "label": null, "model_id": "Qwen/Qwen2.5-0.5B-Instruct", "tokenizer_id": "Qwen/Qwen2.5-0.5B-Instruct", "num_prompts": 256, "request_rate": "inf", "burstiness": 1.0, "max_concurrency": 1, "duration": 228.24286612499998, "completed": 256, "failed": 0, "total_input_tokens": 131072, "total_output_tokens": 32768, "request_throughput": 1.121612273567396, "request_goodput": null, "output_throughput": 143.56637101662668, "total_token_throughput": 717.8318550831334, "max_output_tokens_per_s": 152.0, "max_concurrent_requests": 3, "rtfx": 0.0, "mean_ttft_ms": 58.9748597421913, "median_ttft_ms": 58.521748999964984, "std_ttft_ms": 3.4637904149495857, "p50_ttft_ms": 58.521748999964984, "p95_ttft_ms": 63.53849649985932, "mean_tpot_ms": 6.551345109159686, "median_tpot_ms": 6.5655655157480926, "std_tpot_ms": 0.08603199011451997, "p50_tpot_ms": 6.5655655157480926, "p95_tpot_ms": 6.68668717913429

In [10]:
!nproc; grep -m1 "model name" /proc/cpuinfo; free -g | sed -n 2p; pip list 2>/dev/null | grep -E "^torch "

2
model name	: Intel(R) Xeon(R) CPU @ 2.00GHz
Mem:              12           4           1           0           7           8
torch                                    2.14.0
